# Séance 2 : sélectionner

Objectif : garder seulement les colonnes et les lignes qui répondent à une question, puis trier et ajouter des colonnes calculées.

In [1]:
import pandas as pd

prix = pd.read_csv("donnees/prix_marches.csv")
prix.head(3)

,mois,departement,marche,produit,unite,prix_htg
0,2024-01,Ouest,Croix-des-Bossales,Riz importé,marmite,488.0
1,2024-01,Ouest,Croix-des-Bossales,Riz local,marmite,623.0
2,2024-01,Ouest,Croix-des-Bossales,Maïs moulu,marmite,297.0


## 1. Choisir des colonnes

Une colonne : une Series. Une **liste** de colonnes : un DataFrame (attention aux doubles crochets).

In [2]:
prix[["produit", "prix_htg"]].head(3)

,produit,prix_htg
0,Riz importé,488.0
1,Riz local,623.0
2,Maïs moulu,297.0


## 2. `loc` et `iloc`

- `loc[lignes, colonnes]` : par **étiquettes** (l'index, les noms de colonnes). La borne de fin est **comprise**.
- `iloc[lignes, colonnes]` : par **positions** (0, 1, 2...). La borne de fin est **exclue**, comme pour les listes.

In [3]:
prix.loc[0:2, ["marche", "produit", "prix_htg"]]

,marche,produit,prix_htg
0,Croix-des-Bossales,Riz importé,488.0
1,Croix-des-Bossales,Riz local,623.0
2,Croix-des-Bossales,Maïs moulu,297.0


In [4]:
prix.iloc[0:2, [2, 3, 5]]

,marche,produit,prix_htg
0,Croix-des-Bossales,Riz importé,488.0
1,Croix-des-Bossales,Riz local,623.0


In [5]:
prix.iloc[-1]

mois             2025-12
departement       Nippes
marche         Miragoâne
produit        Spaghetti
unite             paquet
prix_htg           124.0
Name: 2303, dtype: object

## 3. Filtrer avec une condition

Une comparaison sur une colonne donne une Series de booléens : un **masque**. `prix[masque]` garde les lignes où le masque vaut `True`.

In [6]:
masque = prix["departement"] == "Ouest"
masque.head(3)

0    True
1    True
2    True
Name: departement, dtype: bool

In [7]:
prix[masque].shape

(384, 6)

## 4. Combiner des conditions

`&` (et), `|` (ou), `~` (non), et **des parenthèses autour de chaque condition**.

In [8]:
ouest_2025 = prix[(prix["departement"] == "Ouest") & (prix["mois"] >= "2025-01")]
ouest_2025.shape

(192, 6)

In [9]:
prix[(prix["departement"] == "Ouest") | (prix["departement"] == "Nord")].shape

(576, 6)

In [10]:
prix[~(prix["departement"] == "Ouest")].shape

(1920, 6)

Avec `and` à la place de `&`, Python ne sait pas répondre « vrai ou faux » pour 2 304 lignes à la fois :

In [11]:
prix[(prix["departement"] == "Ouest") and (prix["mois"] >= "2025-01")]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

## 5. `isin` et `between`

In [12]:
riz = prix[prix["produit"].isin(["Riz importé", "Riz local"])]
riz["produit"].unique()

<StringArray>
['Riz importé', 'Riz local']
Length: 2, dtype: str

In [13]:
prix[prix["prix_htg"].between(500, 600)].shape

(340, 6)

## 6. Trier

In [14]:
prix.sort_values("prix_htg", ascending=False).head(3)

,mois,departement,marche,produit,unite,prix_htg
2189,2025-11,Grand'Anse,Jérémie,Huile végétale,gallon,2884.0
1789,2025-07,Nord-Ouest,Port-de-Paix,Huile végétale,gallon,2816.0
2285,2025-12,Grand'Anse,Jérémie,Huile végétale,gallon,2784.0


In [15]:
prix.sort_values(["produit", "prix_htg"]).head(3)

,mois,departement,marche,produit,unite,prix_htg
19,2024-01,Artibonite,Gonaïves,Farine de blé,marmite,365.0
139,2024-02,Sud,Les Cayes,Farine de blé,marmite,390.0
91,2024-01,Nippes,Miragoâne,Farine de blé,marmite,391.0


In [16]:
prix.nlargest(3, "prix_htg")[["mois", "marche", "produit", "prix_htg"]]

,mois,marche,produit,prix_htg
2189,2025-11,Jérémie,Huile végétale,2884.0
1789,2025-07,Port-de-Paix,Huile végétale,2816.0
2285,2025-12,Jérémie,Huile végétale,2784.0


## 7. Colonnes calculées

On affecte une nouvelle colonne avec `prix["nom"] = ...`. Le calcul s'applique à toutes les lignes, comme avec NumPy.

In [17]:
prix["prix_usd"] = (prix["prix_htg"] / 131).round(2)
prix["cher"] = prix["prix_htg"] > 1000
prix[["produit", "prix_htg", "prix_usd", "cher"]].head(3)

,produit,prix_htg,prix_usd,cher
0,Riz importé,488.0,3.73,False
1,Riz local,623.0,4.76,False
2,Maïs moulu,297.0,2.27,False


In [18]:
prix["cher"].sum()

np.int64(482)

## 8. Filtrer et choisir les colonnes en une fois

In [19]:
prix.loc[prix["cher"], ["mois", "marche", "produit", "prix_htg"]].head(3)

,mois,marche,produit,prix_htg
5,2024-01,Croix-des-Bossales,Huile végétale,1385.0
13,2024-01,Pétion-Ville,Huile végétale,1396.0
21,2024-01,Gonaïves,Huile végétale,1389.0


## Exercices en binôme

Les énoncés et les résultats attendus sont sur la page de la séance 2.

## Aller plus loin

- La méthode **`query`** : écrivez le filtre « Ouest en 2025 » avec `prix.query(...)`.
- Le **Copy-on-Write** de pandas 3 : pourquoi `prix[prix["cher"]]["prix_htg"] = 0` ne modifie pas `prix` ?
- Le **MultiIndex** : que donne `prix.set_index(["departement", "marche"])` ?

## Sources

- pandas, « Indexing and selecting data » : https://pandas.pydata.org/docs/user_guide/indexing.html
- pandas, « 10 minutes to pandas », partie Selection : https://pandas.pydata.org/docs/user_guide/10min.html#selection
- Wes McKinney, *Python for Data Analysis*, chapitre 5.2 : https://wesmckinney.com/book/